# 🧠 Ecko Networking System
**A personal networking prep tool for the socially anxious founder.**

Run each cell in order. Fill in your own data — the AI generates everything from *your* voice, not a template.

---
**需要準備：** [Groq API key](https://console.groq.com/keys)（免費）

| Step | 做什麼 | 時機 |
|---|---|---|
| 1–2 | 安裝 + API key | 第一次使用 |
| 3–4 | 基本資料 + 第二大腦 | 第一次使用，之後可更新 |
| 5–6 | 新增活動 + 生成準備包 | **活動前** |
| 7–8 | 記錄聯絡人 + 生成 follow-up | **活動後** |

In [ ]:
# @title ⚙️ Step 1：安裝（只需跑一次）
!git clone https://github.com/onlycaxxy/eco-networking 2>/dev/null || git -C eco-networking pull
%cd /content/eco-networking
!pip install -q -r requirements.txt
print('✅ 安裝完成')

In [ ]:
# @title 🔑 Step 2：API Key 設定
# 先在左側邊欄點「🔑 Secrets」，新增一筆：
#   Name:  GROQ_API_KEY
#   Value: 你的 Groq key（gsk_...）
# 記得把「Notebook access」開關打開

from google.colab import userdata
import yaml
from pathlib import Path

try:
    groq_key = userdata.get('GROQ_API_KEY')
    print('✅ Groq API key 讀取成功')
except Exception:
    groq_key = ''
    print('⚠️  找不到 GROQ_API_KEY，請確認 Secrets 設定後重跑')

In [ ]:
# @title 👤 Step 3：你的基本資料
name    = 'Cathy' # @param {type:"string"}
product = 'Ecko'  # @param {type:"string"}
tagline = 'AI journal app，幫助焦慮的思考者找回聲音' # @param {type:"string"}

config = {
    'user': {'name': name, 'product': product, 'tagline': tagline},
    'claude': {
        'supplier': 'Groq',
        'base_url': 'https://api.groq.com/openai/v1',
        'api_key':  groq_key,
        'model':    'llama-3.3-70b-versatile',
    }
}
Path('config.yaml').write_text(yaml.dump(config, allow_unicode=True))
print(f'✅  {name} / {product} — 設定完成')

In [ ]:
# @title 🧠 Step 4：你的第二大腦
# 越真實、越具體，生成的內容就越像你

pitch       = '你對產品最順口的一句話介紹' # @param {type:"string"}
intro       = '你真的會說出口的自我介紹（不超過 75 字）' # @param {type:"string"}
insight     = '你最近在想的一個問題，或你會自然聊起的話題' # @param {type:"string"}
anxiety_tip = '你實際用過有效的社交焦慮應對策略' # @param {type:"string"}

from ecko.db import NetworkingDB
import sqlite3
db = NetworkingDB()
db.init()

conn = sqlite3.connect('events.db')
conn.execute('DELETE FROM brain')
conn.commit(); conn.close()

db.add_brain('pitch',       '我的 pitch',  pitch)
db.add_brain('intro',       '自我介紹',     intro)
db.add_brain('insight',     '破冰話題',     insight)
db.add_brain('anxiety_tip', '焦慮應對策略', anxiety_tip)

print('✅  第二大腦已更新')
for label, val in [('pitch', pitch), ('intro', intro), ('insight', insight), ('anxiety', anxiety_tip)]:
    print(f'   {label:10} → {val[:45]}…')

---
## 活動前 — 生成準備包

In [ ]:
# @title 📅 Step 5：新增活動
event_name     = '活動名稱' # @param {type:"string"}
event_datetime = '2026-06-13 19:00' # @param {type:"string"}
event_type     = 'meetup' # @param ["meetup", "coffee_chat"]
event_location = '台北' # @param {type:"string"}
event_notes    = '參加者輪廓、活動形式、你想達成的目標' # @param {type:"string"}

conn = sqlite3.connect('events.db')
conn.execute('DELETE FROM events')
conn.commit(); conn.close()

event_id = db.add_event(
    name=event_name, datetime=event_datetime,
    type_=event_type, location=event_location, notes=event_notes,
)
print(f'✅  活動已建立（ID {event_id}）')
print(f'   {event_name}  ｜  {event_datetime}  ｜  {event_location}')

In [ ]:
# @title 📨 Step 8：生成 Follow-up 草稿
# 根據對話記錄 + 你想帶給對方的東西，生成三層結構的訊息
# 連結（記得什麼）→ 價值（帶給對方什麼）→ 行動（具體下一步）

followup_contact_id = contact_id # @param {type:"integer"}
platform            = 'LinkedIn' # @param ["LinkedIn", "Email"]
follow_up_intent    = '這次想帶給對方的東西（例：分享文章、提議試用、介紹某人、你後來想到的 insight）' # @param {type:"string"}

target = next((c for c in db.list_contacts() if c.id == followup_contact_id), None)

if not target:
    print(f'❌  找不到 ID {followup_contact_id}')
else:
    brain = {t: db.get_brain(t) for t in ('pitch', 'intro', 'insight', 'anxiety_tip')}
    print(f'⏳  為 {target.name} 生成 {platform} 訊息...')
    draft = PrepGenerator(config).generate_followup(target, brain, platform, follow_up_intent)
    display(Markdown(
        f'### 📨 {platform} Follow-up — {target.name}\n\n'
        f'---\n\n{draft}\n\n---\n\n'
        f'> 💬 聊了：{(target.notes or "")[:80]}…\n\n'
        f'> 🎯 想帶給對方：{follow_up_intent[:60]}'
    ))

---
## 活動後 — 記錄 & Follow-up

In [ ]:
# @title 📝 Step 7：記錄聯絡人
# 每個人填一次，重複執行這個 cell 可以新增多人

contact_name    = '對方姓名' # @param {type:"string"}
contact_role    = '對方在做什麼' # @param {type:"string"}
contact_info    = 'email 或 LinkedIn' # @param {type:"string"}
contact_notes   = '我們聊了什麼？越具體越好，follow-up 草稿會用到這裡的內容' # @param {type:"string"}
follow_up_days  = 3 # @param {type:"integer"}

from datetime import datetime, timedelta
follow_up_by = (datetime.now() + timedelta(days=follow_up_days)).strftime('%Y-%m-%d')

contact_id = db.add_contact(
    event_id=event_id,
    name=contact_name,
    role=contact_role,
    contact=contact_info,
    notes=contact_notes,
    follow_up_by=follow_up_by,
)
print(f'✅  已記錄：{contact_name}（ID {contact_id}）')
print(f'   Follow-up deadline：{follow_up_by}')
print(f'   備註：{contact_notes[:60]}…')

In [ ]:
# @title 📨 Step 8：生成 Follow-up 草稿
# 根據你記的對話備註，生成一則可以直接傳送的訊息

followup_contact_id = contact_id # @param {type:"integer"}
platform = 'LinkedIn' # @param ["LinkedIn", "Email"]

from ecko.models import Contact

target = db.list_contacts()
target = next((c for c in target if c.id == followup_contact_id), None)

if not target:
    print(f'❌  找不到 ID {followup_contact_id}')
else:
    brain = {t: db.get_brain(t) for t in ('pitch', 'intro', 'insight', 'anxiety_tip')}
    print(f'⏳  為 {target.name} 生成 {platform} 訊息...')
    draft = PrepGenerator(config).generate_followup(target, brain, platform)
    display(Markdown(f'### 📨 {platform} Follow-up — {target.name}\n\n---\n\n{draft}\n\n---\n\n> 聊了：{target.notes[:80]}…'))